# 03 ? Validaci?n del modelo relacional

## Objetivo y alcance
Evaluar unicidad de los identificadores candidatos y contrastar las relaciones sugeridas por nombres de columna exactamente iguales entre m?dulos. Se reportan m?tricas agregadas de cobertura y posibles registros sin correspondencia. No se crean joins ni tablas derivadas.

## M?todo y criterios
- Se leen como texto los CSV descritos en `source_schema.json`, preservando codificaci?n, separador y encabezados fuente. Vac?os exactos y celdas solo con espacios se cuentan aparte; ambos se excluyen de las comparaciones.
- Un candidato se observa como ?nico solo si cada fila tiene valor no vac?o y todos los valores son distintos. Es una propiedad emp?rica del archivo, no una confirmaci?n sem?ntica de clave primaria.
- Solo se comparan candidatos con nombre exacto coincidente entre tablas. Si solo un lado es ?nico, se usa provisionalmente como lado padre y se eval?an coincidencias por pertenencia, sin materializar un join. Si hay filas hijas sin valor padre, la integridad referencial candidata se rechaza; sin direcci?n ?nica queda pendiente.
- La etiqueta `validada` describe ?nicamente que el candidato ?nico cubre los valores no vac?os del otro lado. La interpretaci?n sem?ntica y su respaldo documental siguen pendientes.

## Privacidad y limitaciones
Los valores de microdatos se mantienen solo en memoria para las comparaciones. Los reportes incluyen rutas, nombres de columnas y conteos; nunca valores fuente ni hashes. Igualdad de nombre y coincidencia emp?rica no demuestran por s? solas que dos campos representen la misma entidad.

In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontr? pyproject.toml desde el directorio actual")
src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from mode_choice.relational_validation import validate_relational_candidates, write_relational_reports

# Completa la lectura y validaci?n antes de reemplazar los reportes anteriores.
candidate_results, relationship_results = validate_relational_candidates(PROJECT_ROOT)
candidate_path, relationship_path = write_relational_reports(PROJECT_ROOT, candidate_results, relationship_results)

candidates = pd.DataFrame(candidate_results)
relationships = pd.DataFrame(relationship_results)
print(f"Candidatos auditados: {len(candidates)}")
print(f"Relaciones candidatas comparadas: {len(relationships)}")
print(f"Reportes: {candidate_path.relative_to(PROJECT_ROOT)}; {relationship_path.relative_to(PROJECT_ROOT)}")
display(candidates)
display(relationships)

Candidatos auditados: 21
Relaciones candidatas comparadas: 11
Reportes: docs\data_dictionary\relational_candidate_audit.csv; docs\data_dictionary\relational_relationship_audit.csv


,source_path,source_name,row_count,empty_count,whitespace_only_count,distinct_nonempty_count,repeated_nonempty_count,observed_unique_nonempty,assessment,candidate_status
0,data/sources/raw/Database/a. Modulo hogares.csv,cod_viv,22755,0,0,21669,1086,False,repetition_or_missing_observed,name_based_candidate_only; semantic_role_uncon...
1,data/sources/raw/Database/a. Modulo hogares.csv,cod_hog,22755,0,0,22755,0,True,unique_observed,name_based_candidate_only; semantic_role_uncon...
2,data/sources/raw/Database/a. Modulo hogares.csv,key_hg,22755,0,0,22755,0,True,unique_observed,name_based_candidate_only; semantic_role_uncon...
3,data/sources/raw/Database/b. Modulo vehiculos.csv,cod_vh,17392,0,0,17392,0,True,unique_observed,name_based_candidate_only; semantic_role_uncon...
4,data/sources/raw/Database/b. Modulo vehiculos.csv,cod_viv,17392,0,0,12269,5123,False,repetition_or_missing_observed,name_based_candidate_only; semantic_role_uncon...
5,data/sources/raw/Database/b. Modulo vehiculos.csv,cod_hog,17392,0,0,12582,4810,False,repetition_or_missing_observed,name_based_candidate_only; semantic_role_uncon...
6,data/sources/raw/Database/b. Modulo vehiculos.csv,key_hg,17392,0,0,12582,4810,False,repetition_or_missing_observed,name_based_candidate_only; semantic_role_uncon...
7,data/sources/raw/Database/c. Modulo personas.csv,cod_per,67556,0,0,67556,0,True,unique_observed,name_based_candidate_only; semantic_role_uncon...
8,data/sources/raw/Database/c. Modulo personas.csv,key_hg,67556,0,0,22755,44801,False,repetition_or_missing_observed,name_based_candidate_only; semantic_role_uncon...
9,data/sources/raw/Database/c. Modulo personas.csv,key_persona,67556,0,0,67556,0,True,unique_observed,name_based_candidate_only; semantic_role_uncon...


,candidate_name,parent_source,parent_column,child_source,child_column,parent_unique_nonempty_values,child_distinct_nonempty_values,distinct_values_present_in_both,child_distinct_values_without_parent,child_empty_rows,child_whitespace_only_rows,child_rows_with_match,child_rows_without_match,parent_distinct_values_without_child,validation_status,status_note,source_a,source_b
0,cod_viv,,cod_viv,,cod_viv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,direction_ambiguous_or_no_unique_side,data/sources/raw/Database/a. Modulo hogares.csv,data/sources/raw/Database/b. Modulo vehiculos.csv
1,cod_hog,data/sources/raw/Database/a. Modulo hogares.csv,cod_hog,data/sources/raw/Database/b. Modulo vehiculos.csv,cod_hog,22755.0,12582.0,12582.0,0.0,0.0,0.0,17392.0,0.0,10173.0,validada,empirical referential-integrity signal; semant...,NaN,NaN
2,key_hg,data/sources/raw/Database/a. Modulo hogares.csv,key_hg,data/sources/raw/Database/b. Modulo vehiculos.csv,key_hg,22755.0,12582.0,12582.0,0.0,0.0,0.0,17392.0,0.0,10173.0,validada,empirical referential-integrity signal; semant...,NaN,NaN
3,key_hg,data/sources/raw/Database/a. Modulo hogares.csv,key_hg,data/sources/raw/Database/c. Modulo personas.csv,key_hg,22755.0,22755.0,22755.0,0.0,0.0,0.0,67556.0,0.0,0.0,validada,empirical referential-integrity signal; semant...,NaN,NaN
4,key_hg,data/sources/raw/Database/a. Modulo hogares.csv,key_hg,data/sources/raw/Database/d. Modulo viajes.csv,key_hg,22755.0,21508.0,21508.0,0.0,0.0,0.0,100174.0,0.0,1247.0,validada,empirical referential-integrity signal; semant...,NaN,NaN
5,key_hg,,key_hg,,key_hg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,direction_ambiguous_or_no_unique_side,data/sources/raw/Database/b. Modulo vehiculos.csv,data/sources/raw/Database/c. Modulo personas.csv
6,key_hg,,key_hg,,key_hg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,direction_ambiguous_or_no_unique_side,data/sources/raw/Database/b. Modulo vehiculos.csv,data/sources/raw/Database/d. Modulo viajes.csv
7,key_hg,,key_hg,,key_hg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,direction_ambiguous_or_no_unique_side,data/sources/raw/Database/c. Modulo personas.csv,data/sources/raw/Database/d. Modulo viajes.csv
8,cod_pers,,cod_pers,,cod_pers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,direction_ambiguous_or_no_unique_side,data/sources/raw/Database/d. Modulo viajes.csv,data/sources/raw/Database/e. Modulo etapas.csv
9,cod_vj,data/sources/raw/Database/d. Modulo viajes.csv,cod_vj,data/sources/raw/Database/e. Modulo etapas.csv,cod_vj,100174.0,100174.0,100174.0,0.0,0.0,0.0,110881.0,0.0,0.0,validada,empirical referential-integrity signal; semant...,NaN,NaN
